#Dataset

# Overview

1. Read the raw data (e.g., u.data) from github
2. Sort and extract the data (e.g., 100-by-100 matrix) 
  * Use "create_data" function
3. Create the k-fold dataset (e.g., k=10) 
  * Use "k_fold_data" function
4. Save the k data files 

In [10]:
import numpy as np

In [19]:
# Set parameters
n_items = 100
n_users = 300
k = 10 

In [20]:
# Create n-by-m matrix (input: df, output: new_df)
def create_data(df, n, m):
  
  pivot_df = df.pivot_table(index="item_id", columns="user_id",values="rating", aggfunc='mean')
  
  n_row = pivot_df.shape[0]
  n_col = pivot_df.shape[1]
  Obs_ind = np.where(pivot_df.notnull())    # Row and column indices for the observed entries of "Mdat"
  num_Obs = len(Obs_ind[0])           # The number of observed entries of "df"
  sparsity = 1 - num_Obs / (n_row * n_col)
  #print(n_row,n_col,num_Obs,sparsity)
  print(f'(Original) matrix sparsity: {sparsity:f}')

  #Sorting and extracting 
  new_df1 = pivot_df.copy()
  new_df1['count'] = new_df1.count(axis=1)
  new_df1_sort = new_df1.sort_values('count', ascending=False)
  new_df2 = new_df1_sort.drop("count", axis=1, inplace=False)
  new_df3 = new_df2.transpose()
  new_df3['count'] = new_df3.count(axis=1)
  new_df3_sort = new_df3.sort_values('count', ascending=False)  
  final_df = new_df3_sort.drop("count", axis=1, inplace=False)
  final_data = final_df.transpose()
  new_df = final_data.iloc[0:n,0:m]
  #print(new_df)
  
  n_row = new_df.shape[0]
  n_col = new_df.shape[1]
  Obs_ind = np.where(new_df.notnull())    # Row and column indices for the observed entries of "Mdat"
  num_Obs = len(Obs_ind[0])           # The number of observed entries of "df"
  sparsity = 1 - num_Obs / (n_row * n_col)
  #print(n_row,n_col,num_Obs,sparsity)
  sparsity = 1 - num_Obs / (m * n)

  print(f'(Extracted) matrix sparsity: {sparsity:f}')


  return new_df

In [21]:
import numpy as np
import pandas as pd
from pandas.core.describe import DataFrameDescriber
# from google.colab import files
url = "https://raw.githubusercontent.com/park61/imputation/main/u.data"
# creating a pandas data frame
colnames = ['user_id', 'item_id', 'rating', 'timestamp']
df = pd.read_csv(url,sep='\t', names=colnames, header=None)
#df = pd.read_csv("test.csv")#,sep='\t', names=colnames, header=None)
new_df = create_data(df, n_items, n_users)
new_df.to_csv(str(n_items)+'-by-'+str(n_users)+'_'+'original_matrix.csv')
# files.download(str(n_items)+'-by-'+str(n_users)+'_'+'original_matrix.csv')
display(new_df)

(Original) matrix sparsity: 0.936953
(Extracted) matrix sparsity: 0.420200


user_id,405,655,13,450,276,416,537,303,234,393,...,292,671,236,263,250,645,661,38,391,361
item_id,,,,,,,,,,,,,,,,,,,,,
50,5.0,4.0,5.0,5.0,5.0,5.0,4.0,5.0,4.0,5.0,...,4.0,5.0,3.0,5.0,5.0,4.0,5.0,NaN,4.0,5.0
258,NaN,2.0,4.0,4.0,5.0,5.0,4.0,4.0,2.0,4.0,...,NaN,5.0,NaN,3.0,4.0,3.0,4.0,NaN,3.0,3.0
100,NaN,3.0,5.0,4.0,5.0,5.0,4.0,5.0,4.0,1.0,...,5.0,NaN,3.0,5.0,5.0,NaN,NaN,NaN,4.0,5.0
181,5.0,3.0,5.0,4.0,5.0,5.0,2.0,5.0,3.0,4.0,...,4.0,5.0,4.0,4.0,4.0,4.0,5.0,NaN,NaN,NaN
294,NaN,3.0,2.0,4.0,4.0,4.0,1.0,4.0,3.0,4.0,...,NaN,NaN,2.0,2.0,1.0,NaN,4.0,5.0,2.0,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
95,3.0,NaN,5.0,3.0,5.0,3.0,1.0,5.0,3.0,4.0,...,NaN,NaN,NaN,5.0,5.0,NaN,5.0,5.0,NaN,NaN
8,4.0,3.0,4.0,NaN,4.0,5.0,NaN,5.0,5.0,3.0,...,NaN,NaN,NaN,NaN,NaN,NaN,5.0,NaN,3.0,NaN
427,5.0,4.0,5.0,5.0,5.0,5.0,4.0,4.0,4.0,NaN,...,NaN,NaN,5.0,NaN,NaN,5.0,4.0,NaN,5.0,NaN


: 

In [14]:
pivot_df = df.pivot_table(index="item_id", columns="user_id",values="rating", aggfunc='mean')
n_row = pivot_df.shape[0]
n_col = pivot_df.shape[1]
print(n_row)
print(n_col)

1682
943


In [15]:
#Create the k-fold datasets 
import random
def k_fold_data(df,k):

  n_row = df.shape[0]
  n_col = df.shape[1]
  Obs_ind = np.where(df.notnull())    # Row and column indices for the observed entries of "Mdat"
  num_Obs = len(Obs_ind[0])           # The number of observed entries of "df"
  sparsity = 1 - num_Obs / (n_row * n_col)
  #print(n_row,n_col,num_Obs,sparsity)
  print(f'(Original) matrix sparsity: {sparsity:f}')

  a = int(num_Obs/k)
  A = k*a + k - num_Obs
  num_Obs_each = np.append(a*np.ones(A), (a+1)*np.ones(k-A)).astype(int)
  #print(num_Obs_each)

  label = np.array(range(num_Obs))

  p = [None] * k
  test_dat = [None] * k

  for i in range(k):

    Tdat = pd.DataFrame.copy(df)
    random.seed(1111)#1->1111
    random.shuffle(label)
    p[i] = label[:num_Obs_each[i]]
    #print(p[i])
    for j in p[i]:
      Tdat.iloc[Obs_ind[0][j], Obs_ind[1][j]] = np.nan
    
    test_dat[i] = Tdat
    #print(test_dat[i])
    #print(len(np.where(test_dat[i].notnull())[0]))
    sparsity = 1 -  len(np.where(test_dat[i].notnull())[0]) / (test_dat[i].shape[0] * test_dat[i].shape[1])
    print(str(i)+"th data sparsity = "+str(sparsity))

  return test_dat


In [16]:
test_dat = k_fold_data(new_df,k)
#display(test_dat[0])
display(test_dat[k-1])

(Original) matrix sparsity: 0.420200
0th data sparsity = 0.47816666666666663
1th data sparsity = 0.47816666666666663
2th data sparsity = 0.47816666666666663
3th data sparsity = 0.47816666666666663
4th data sparsity = 0.47816666666666663
5th data sparsity = 0.47816666666666663
6th data sparsity = 0.47819999999999996
7th data sparsity = 0.47819999999999996
8th data sparsity = 0.47819999999999996
9th data sparsity = 0.47819999999999996


user_id,405,655,13,450,276,416,537,303,234,393,...,292,671,236,263,250,645,661,38,391,361
item_id,,,,,,,,,,,,,,,,,,,,,
50,5.0,4.0,5.0,5.0,5.0,NaN,4.0,5.0,NaN,5.0,...,4.0,5.0,3.0,5.0,5.0,4.0,5.0,NaN,4.0,5.0
258,NaN,2.0,4.0,4.0,5.0,5.0,4.0,4.0,2.0,4.0,...,NaN,5.0,NaN,3.0,4.0,3.0,4.0,NaN,3.0,3.0
100,NaN,3.0,5.0,4.0,5.0,5.0,4.0,5.0,4.0,1.0,...,5.0,NaN,3.0,NaN,5.0,NaN,NaN,NaN,4.0,5.0
181,5.0,3.0,5.0,4.0,5.0,5.0,2.0,5.0,3.0,NaN,...,4.0,5.0,NaN,4.0,4.0,4.0,NaN,NaN,NaN,NaN
294,NaN,3.0,2.0,4.0,4.0,NaN,1.0,4.0,3.0,4.0,...,NaN,NaN,2.0,NaN,1.0,NaN,NaN,NaN,2.0,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
95,3.0,NaN,5.0,3.0,5.0,3.0,1.0,5.0,3.0,4.0,...,NaN,NaN,NaN,NaN,5.0,NaN,5.0,5.0,NaN,NaN
8,4.0,3.0,4.0,NaN,4.0,5.0,NaN,5.0,5.0,3.0,...,NaN,NaN,NaN,NaN,NaN,NaN,5.0,NaN,NaN,NaN
427,5.0,4.0,NaN,5.0,5.0,NaN,4.0,4.0,4.0,NaN,...,NaN,NaN,5.0,NaN,NaN,5.0,4.0,NaN,5.0,NaN


In [17]:
#Checking invalid dataset 
for i in range(k):
  print(i, min((1-test_dat[i].isna()).sum(axis=0)), min((1-test_dat[i].isna()).sum(axis=1)))

0 20 70
1 16 70
2 20 62
3 21 70
4 18 66
5 16 65
6 22 69
7 18 68
8 20 66
9 19 70


# 4. Save data

In [18]:
for i in range(k):
  if i<9:
    filename = str(n_items)+'-by-'+str(n_users)+'_'+'10_fold_test_data0'+str(i+1)+'.csv'
  else:
    filename = str(n_items)+'-by-'+str(n_users)+'_'+'10_fold_test_data'+str(i+1)+'.csv'
  test_dat[i].to_csv(filename)